# 03 · Module 层级导航

> **覆盖的类/函数**：`children`, `named_children`, `modules`, `named_modules`, `get_submodule`  
> **PyTorch 源码**：[torch/nn/modules/module.py](https://github.com/pytorch/pytorch/blob/main/torch/nn/modules/module.py)  
> **运行环境**：Python 3.9+, PyTorch 2.0+

本 Notebook 聚焦于 `nn.Module` 如何通过 `_modules` 字典构成一棵树，以及遍历这棵树的 5 个核心方法。

---

## Part A: 源码阅读

先导入模块并构建示例嵌套模型，用于后续所有实验：

In [1]:
import torch
import torch.nn as nn
import inspect

print(f'PyTorch 版本: {torch.__version__}')

# 构建一个示例嵌套模型，方便后续实验
class SubBlock(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)
        self.bn = nn.BatchNorm1d(out_dim)
        self.relu = nn.ReLU()

class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(1000, 128)
        self.block1 = SubBlock(128, 256)
        self.block2 = SubBlock(256, 512)
        self.dropout = nn.Dropout(0.5)
        self.head = nn.Linear(512, 10)

model = MyModel()
print(model)

PyTorch 版本: 2.8.0
MyModel(
  (embedding): Embedding(1000, 128)
  (block1): SubBlock(
    (linear): Linear(in_features=128, out_features=256, bias=True)
    (bn): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU()
  )
  (block2): SubBlock(
    (linear): Linear(in_features=256, out_features=512, bias=True)
    (bn): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU()
  )
  (dropout): Dropout(p=0.5, inplace=False)
  (head): Linear(in_features=512, out_features=10, bias=True)
)


### A.1 `named_modules` — DFS 递归遍历引擎

`named_modules()` 是整个层级导航系统的核心。`modules()`、`named_parameters()`、`named_buffers()` 都依赖它。
它的实现是经典的 **DFS pre-order**（深度优先，先根遍历）：先产出当前节点，再递归产出子节点。

In [2]:
print(inspect.getsource(nn.Module.named_modules))

    def named_modules(
        self,
        memo: Optional[set["Module"]] = None,
        prefix: str = "",
        remove_duplicate: bool = True,
    ):
        r"""Return an iterator over all modules in the network, yielding both the name of the module as well as the module itself.

        Args:
            memo: a memo to store the set of modules already added to the result
            prefix: a prefix that will be added to the name of the module
            remove_duplicate: whether to remove the duplicated module instances in the result
                or not

        Yields:
            (str, Module): Tuple of name and module

        Note:
            Duplicate modules are returned only once. In the following
            example, ``l`` will be returned only once.

        Example::

            >>> l = nn.Linear(2, 2)
            >>> net = nn.Sequential(l, l)
            >>> for idx, m in enumerate(net.named_modules()):
            ...     print(idx, '->', m)

            0 ->

**关键设计点**：

| 特性 | 说明 |
|------|------|
| **DFS pre-order** | 先 `yield prefix, self`，再 `for name, module in self._modules.items()` |
| **生成器 yield from** | 用递归生成器实现懒加载，内存高效 |
| **memo 去重** | 用 `set()` 记录已产出模块的 `id()`，避免共享模块被重复计入 |
| **remove_duplicate** | 默认 `True`，可设为 `False` 允许重复 |
| **prefix 拼接** | 子模块前缀 = `prefix + "." + name`，根模块前缀为 `""` |

### A.2 `modules` — 只取 Module 本身（丢掉名字）

In [3]:
print(inspect.getsource(nn.Module.modules))

    def modules(self) -> Iterator["Module"]:
        r"""Return an iterator over all modules in the network.

        Yields:
            Module: a module in the network

        Note:
            Duplicate modules are returned only once. In the following
            example, ``l`` will be returned only once.

        Example::

            >>> l = nn.Linear(2, 2)
            >>> net = nn.Sequential(l, l)
            >>> for idx, m in enumerate(net.modules()):
            ...     print(idx, '->', m)

            0 -> Sequential(
              (0): Linear(in_features=2, out_features=2, bias=True)
              (1): Linear(in_features=2, out_features=2, bias=True)
            )
            1 -> Linear(in_features=2, out_features=2, bias=True)

        """
        for _, module in self.named_modules():
            yield module



非常简洁 —— 它只是 `named_modules()` 的一个薄包装，丢弃了路径前缀。

### A.3 `named_children` — 只迭代直接子模块（不递归）

In [4]:
print(inspect.getsource(nn.Module.named_children))

    def named_children(self) -> Iterator[tuple[str, "Module"]]:
        r"""Return an iterator over immediate children modules, yielding both the name of the module as well as the module itself.

        Yields:
            (str, Module): Tuple containing a name and child module

        Example::

            >>> # xdoctest: +SKIP("undefined vars")
            >>> for name, module in model.named_children():
            >>>     if name in ['conv4', 'conv5']:
            >>>         print(module)

        """
        memo = set()
        for name, module in self._modules.items():
            if module is not None and module not in memo:
                memo.add(module)
                yield name, module



**区别**：`named_children()` 直接遍历 `self._modules.items()`，**不递归**。
所以 `named_children()` 返回的是模块树的「直接子节点」，而 `named_modules()` 返回的是整棵树的「所有节点」。

> ⚠️ `named_children()` 也有自己的 `memo` set 用于去重 —— 虽然直接子模块中重复的情况罕见，但设计上与 `named_modules()` 保持一致。

### A.4 `children` — `named_children` 的简化版

In [5]:
print(inspect.getsource(nn.Module.children))

    def children(self) -> Iterator["Module"]:
        r"""Return an iterator over immediate children modules.

        Yields:
            Module: a child module
        """
        for _name, module in self.named_children():
            yield module



### A.5 `get_submodule` — 点分隔路径查找

按 `"block1.linear"` 这样的路径快速定位到嵌套子模块。时间复杂度 O(depth)，远优于遍历 `named_modules()` 的 O(N)。

In [6]:
print(inspect.getsource(nn.Module.get_submodule))

    def get_submodule(self, target: str) -> "Module":
        """Return the submodule given by ``target`` if it exists, otherwise throw an error.

        For example, let's say you have an ``nn.Module`` ``A`` that
        looks like this:

        .. code-block:: text

            A(
                (net_b): Module(
                    (net_c): Module(
                        (conv): Conv2d(16, 33, kernel_size=(3, 3), stride=(2, 2))
                    )
                    (linear): Linear(in_features=100, out_features=200, bias=True)
                )
            )

        (The diagram shows an ``nn.Module`` ``A``. ``A`` which has a nested
        submodule ``net_b``, which itself has two submodules ``net_c``
        and ``linear``. ``net_c`` then has a submodule ``conv``.)

        To check whether or not we have the ``linear`` submodule, we
        would call ``get_submodule("net_b.linear")``. To check whether
        we have the ``conv`` submodule, we would call
        ``get_su

### A.6 `_named_members` — `named_parameters` / `named_buffers` 的共享遍历引擎

（在 Notebook 02 中已详细讲解，这里简要回顾其与层级导航的关系）

In [7]:
print(inspect.getsource(nn.Module._named_members))

    def _named_members(
        self, get_members_fn, prefix="", recurse=True, remove_duplicate: bool = True
    ):
        r"""Help yield various names + members of modules."""
        memo = set()
        modules = (
            self.named_modules(prefix=prefix, remove_duplicate=remove_duplicate)
            if recurse
            else [(prefix, self)]
        )
        for module_prefix, module in modules:
            members = get_members_fn(module)
            for k, v in members:
                if v is None or v in memo:
                    continue
                if remove_duplicate:
                    memo.add(v)
                name = module_prefix + ("." if module_prefix else "") + k
                yield name, v



`_named_members` 在 `recurse=True` 时调用 `self.named_modules()` 来遍历所有子模块，然后用 `get_members_fn` 提取每个模块的参数/Buffer。
这就是为什么理解 `named_modules()` 如此重要 —— 它是整个层级遍历系统的根基。

---

## Part B: 机制分析

### B.1 DFS Pre-order 遍历顺序

`named_modules()` 遍历顺序是：**根 → 第一个子模块及其全部后代 → 第二个子模块及其全部后代 → ...**

对于我们的 MyModel：

```
MyModel                           ← yield 0: ''
├── embedding (Embedding)         ← yield 1: 'embedding'
├── block1 (SubBlock)             ← yield 2: 'block1'
│   ├── linear (Linear)           ← yield 3: 'block1.linear'
│   ├── bn (BatchNorm1d)          ← yield 4: 'block1.bn'
│   └── relu (ReLU)               ← yield 5: 'block1.relu'
├── block2 (SubBlock)             ← yield 6: 'block2'
│   ├── linear (Linear)           ← yield 7: 'block2.linear'
│   ├── bn (BatchNorm1d)          ← yield 8: 'block2.bn'
│   └── relu (ReLU)               ← yield 9: 'block2.relu'
├── dropout (Dropout)             ← yield 10: 'dropout'
└── head (Linear)                 ← yield 11: 'head'
```

> 💡 这个顺序恰好也保证了 `model.to(device)` 和 `model.parameters()` 的确定性遍历顺序。

### B.2 `children()` vs `modules()` 对比

| 维度 | `children()` | `modules()` |
|------|-------------|----------|
| 范围 | 仅直接子模块 | 递归所有后代 |
| 返回类型 | `Iterator[Module]` | `Iterator[Module]` |
| 是否包含自身 | ❌ 不包含 | ✅ 包含（第一个产出） |
| 底层实现 | `named_children()` → `_modules.items()` | `named_modules()` → DFS 递归 |
| 典型用途 | 遍历网络层 | 权重初始化、冻结子网络 |

**直观理解**：
- `model.children()` ≈ `list(model._modules.values())`
- `model.modules()` ≈ `[model] + 递归展开所有 children`

### B.3 去重机制：共享子模块

当同一个 Module 实例作为多个父模块的子模块时，`named_modules()` 默认只会产出它一次。

去重通过 `memo` set 实现：每次 `yield` 前 `memo.add(self)`，下次遇到同一个 `id()` 时跳过。

> ⚠️ **注意**：`named_children()` 和 `named_modules()` 各自维护独立的 `memo`，所以去重范围仅限单次调用。

### B.4 `get_submodule` 的实现策略

`get_submodule(target)` 使用 **逐层 `getattr`** 的方式，而非遍历 `named_modules()`：

1. 按 `.` 分割路径为 `atoms`
2. 从 `self` 开始，对每个 `item` 调用 `getattr(mod, item)`
3. 检查结果是否为 `nn.Module` 实例
4. 任一步失败则抛出 `AttributeError`

每步都依赖 `getattr` → `__getattr__` 从 `_modules` 中查找，所以路径必须完全匹配 `_modules` 的键名。
对于 Parameter 名（如 `weight`）会失败，因为 `getattr` 返回的是 `Parameter` 而非 `Module`。

---

## Part C: 动手实验

### 实验 1：验证 DFS Pre-order 遍历顺序

In [8]:
# 用 named_modules 打印遍历顺序
print('=== named_modules() DFS pre-order 遍历顺序 ===')
for idx, (name, mod) in enumerate(model.named_modules()):
    indent = '  ' * name.count('.') if name else ''
    print(f'{idx:2d}: {indent}{name or "(root)"} -> {type(mod).__name__}')

print(f'总模块数: {idx + 1}')

=== named_modules() DFS pre-order 遍历顺序 ===
 0: (root) -> MyModel
 1: embedding -> Embedding
 2: block1 -> SubBlock
 3:   block1.linear -> Linear
 4:   block1.bn -> BatchNorm1d
 5:   block1.relu -> ReLU
 6: block2 -> SubBlock
 7:   block2.linear -> Linear
 8:   block2.bn -> BatchNorm1d
 9:   block2.relu -> ReLU
10: dropout -> Dropout
11: head -> Linear
总模块数: 12


对照我们在 Part B.1 中画的树结构图，验证遍历顺序是否完全匹配。

In [9]:
# 验证 modules() 与 named_modules() 的对应关系
modules_list = list(model.modules())
named_modules_list = list(model.named_modules())

print(f'modules() 返回数量: {len(modules_list)}')
print(f'named_modules() 返回数量: {len(named_modules_list)}')
print(f'数量一致: {len(modules_list) == len(named_modules_list)}')

# modules() 就是 named_modules() 去掉名字
modules_from_named = [mod for _, mod in named_modules_list]
print(f'modules() 元素 == named_modules() 去掉名字: {all(a is b for a, b in zip(modules_list, modules_from_named))}')

# 第一个元素是根模块自身
print(f'第一个模块是 model 自身: {modules_list[0] is model}')
print(f'最后一个模块: {type(modules_list[-1]).__name__} (head)')

modules() 返回数量: 12
named_modules() 返回数量: 12
数量一致: True
modules() 元素 == named_modules() 去掉名字: True
第一个模块是 model 自身: True
最后一个模块: Linear (head)


### 实验 2：`children()` vs `modules()` — 直接子模块 vs 全部后代

In [10]:
# children 只返回直接子模块（1 层深）
print('=== named_children() — 仅直接子模块 ===')
for name, child in model.named_children():
    grandchildren = [n for n, _ in child.named_children()]
    extra = f' (它还有子模块: {grandchildren})' if grandchildren else ''
    print(f'  {name}: {type(child).__name__}{extra}')

print(f'直接子模块数 (children): {len(list(model.children()))}')
print(f'全部后代数 (modules):   {len(list(model.modules()))}')  # 包含自身
print(f'全部后代不含自身:       {len(list(model.modules())) - 1}')

=== named_children() — 仅直接子模块 ===
  embedding: Embedding
  block1: SubBlock (它还有子模块: ['linear', 'bn', 'relu'])
  block2: SubBlock (它还有子模块: ['linear', 'bn', 'relu'])
  dropout: Dropout
  head: Linear
直接子模块数 (children): 5
全部后代数 (modules):   12
全部后代不含自身:       11


In [11]:
# 直观对比表格
print(f'{"方法":<25} {"返回数量":<10} {"包含自身":<10} {"递归":<10}')
print('-' * 55)
print(f'{"children()":<25} {len(list(model.children())):<10} {"否":<10} {"否":<10}')
print(f'{"modules()":<25} {len(list(model.modules())):<10} {"是":<10} {"是":<10}')
print(f'{"named_children()":<25} {len(list(model.named_children())):<10} {"否":<10} {"否":<10}')
print(f'{"named_modules()":<25} {len(list(model.named_modules())):<10} {"是":<10} {"是":<10}')

# 重要细节：modules() 包含根模块自身
print()
print("关键差异总结:")
print(f"  modules()[0] is model:  {list(model.modules())[0] is model}")
print(f"  model in children():    {model in list(model.children())}")


方法                        返回数量       包含自身       递归        
-------------------------------------------------------
children()                5          否          否         
modules()                 12         是          是         
named_children()          5          否          否         
named_modules()           12         是          是         

关键差异总结:
  modules()[0] is model:  True
  model in children():    False


### 实验 3：去重行为 — 共享子模块

In [12]:
# 构造共享子模块的场景
shared_linear = nn.Linear(10, 10)

class SharedNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.branch_a = shared_linear
        self.branch_b = shared_linear  # 同一个实例！
        self.other = nn.ReLU()

shared_net = SharedNet()

# remove_duplicate=True（默认）
print('=== remove_duplicate=True (默认) ===')
for name, mod in shared_net.named_modules(remove_duplicate=True):
    print(f'  {name or "(root)":20s} -> {type(mod).__name__}  (id={id(mod)})')
print(f'总模块数(去重): {len(list(shared_net.named_modules()))}')

# remove_duplicate=False
print(f'\n=== remove_duplicate=False ===')
for name, mod in shared_net.named_modules(remove_duplicate=False):
    print(f'  {name or "(root)":20s} -> {type(mod).__name__}  (id={id(mod)})')
print(f'总模块数(不去重): {len(list(shared_net.named_modules(remove_duplicate=False)))}')

# 验证去重效果
all_names = [n for n, _ in shared_net.named_modules()]
print(f'\n去重后模块名: {all_names}')
print(f'branch_b 出现了吗? {"branch_b" in all_names}')
print('说明: shared_linear 在 branch_a 下已被产出，branch_b 被去重跳过')

=== remove_duplicate=True (默认) ===
  (root)               -> SharedNet  (id=4497211248)
  branch_a             -> Linear  (id=4497072576)
  other                -> ReLU  (id=4497210288)
总模块数(去重): 3

=== remove_duplicate=False ===
  (root)               -> SharedNet  (id=4497211248)
  branch_a             -> Linear  (id=4497072576)
  branch_b             -> Linear  (id=4497072576)
  other                -> ReLU  (id=4497210288)
总模块数(不去重): 4

去重后模块名: ['', 'branch_a', 'other']
branch_b 出现了吗? False
说明: shared_linear 在 branch_a 下已被产出，branch_b 被去重跳过


In [13]:
# 深入：查看去重对参数计数的影响
print('=== 去重对 named_parameters() 的影响 ===')
# 去重（默认）
params_dedup = list(shared_net.named_parameters(remove_duplicate=True))
print(f'去重: {len(params_dedup)} 个参数')
for name, _ in params_dedup:
    print(f'  {name}')

# 不去重
params_nodedup = list(shared_net.named_parameters(remove_duplicate=False))
print(f'不去重: {len(params_nodedup)} 个参数')
for name, _ in params_nodedup:
    print(f'  {name}')

print(f'\n⚠️ 如果去掉去重，shared_linear 的参数会出现两次!')
print(f'   这会导致 optimizer 对同一参数做两次更新，破坏训练正确性。')

=== 去重对 named_parameters() 的影响 ===
去重: 2 个参数
  branch_a.weight
  branch_a.bias
不去重: 4 个参数
  branch_a.weight
  branch_a.bias
  branch_b.weight
  branch_b.bias

⚠️ 如果去掉去重，shared_linear 的参数会出现两次!
   这会导致 optimizer 对同一参数做两次更新，破坏训练正确性。


### 实验 4：`get_submodule` 路径查找与错误处理

In [14]:
# 成功查找
print('=== get_submodule 路径查找 ===')
results = [
    ('"" (空字符串)', ''),
    ('"block1"', 'block1'),
    ('"block1.linear"', 'block1.linear'),
    ('"block2.bn"', 'block2.bn'),
    ('"head"', 'head'),
]
for label, path in results:
    sub = model.get_submodule(path)
    print(f'  get_submodule({label:25s}) -> {type(sub).__name__}')

# 验证 get_submodule("") 返回自身
print(f'\nget_submodule("") is model: {model.get_submodule("") is model}')

=== get_submodule 路径查找 ===
  get_submodule("" (空字符串)                ) -> MyModel
  get_submodule("block1"                 ) -> SubBlock
  get_submodule("block1.linear"          ) -> Linear
  get_submodule("block2.bn"              ) -> BatchNorm1d
  get_submodule("head"                   ) -> Linear

get_submodule("") is model: True


In [15]:
# 错误情况 1：不存在的属性
print('=== 错误情况 1: 路径不存在 ===')
try:
    model.get_submodule("block3")
except AttributeError as e:
    print(f'  get_submodule("block3")')
    print(f'  {type(e).__name__}: {e}')

# 错误情况 2：路径中间不存在
print('\n=== 错误情况 2: 深层路径不存在 ===')
try:
    model.get_submodule("block1.attention.qkv")
except AttributeError as e:
    print(f'  get_submodule("block1.attention.qkv")')
    print(f'  {type(e).__name__}: {e}')

# 错误情况 3：路径指向 Parameter 而非 Module
print('\n=== 错误情况 3: 路径指向 Parameter 而非 Module ===')
try:
    model.get_submodule("block1.linear.weight")  # weight 是 Parameter
except AttributeError as e:
    print(f'  get_submodule("block1.linear.weight")')
    print(f'  {type(e).__name__}: {e}')

# 错误情况 4：路径包含空字符串
print('\n=== 错误情况 4: 路径包含空字符串 ===')
try:
    model.get_submodule("block1..linear")  # 双点 → 空 atom
except AttributeError as e:
    print(f'  get_submodule("block1..linear")')
    print(f'  空字符串作为模块名通常不存在，触发 {type(e).__name__}')

=== 错误情况 1: 路径不存在 ===
  get_submodule("block3")
  AttributeError: MyModel has no attribute `block3`

=== 错误情况 2: 深层路径不存在 ===
  get_submodule("block1.attention.qkv")
  AttributeError: SubBlock has no attribute `attention`

=== 错误情况 3: 路径指向 Parameter 而非 Module ===
  get_submodule("block1.linear.weight")
  AttributeError: `weight` is not an nn.Module

=== 错误情况 4: 路径包含空字符串 ===
  get_submodule("block1..linear")
  空字符串作为模块名通常不存在，触发 AttributeError


### 实验 5：手动实现模块树可视化 + 理解 `_named_members` 的 `recurse` 参数

In [16]:
# 自定义函数：打印模块树（模拟 named_modules 的 DFS 遍历）
def print_module_tree(module, prefix='', name='', is_last=True, is_root=True):
    """递归打印模块树结构，用树形字符展示层级关系"""
    if is_root:
        print(f'{type(module).__name__} (root)')
    else:
        connector = '└── ' if is_last else '├── '
        print(f'{prefix}{connector}{name}: {type(module).__name__}')
    
    children = list(module.named_children())
    for i, (child_name, child) in enumerate(children):
        is_last_child = (i == len(children) - 1)
        if is_root:
            new_prefix = ''
        else:
            new_prefix = prefix + ('    ' if is_last else '│   ')
        print_module_tree(child, new_prefix, child_name, is_last_child, is_root=False)

print('=== MyModel 模块树（通过 named_children 递归构建）===')
print_module_tree(model)

=== MyModel 模块树（通过 named_children 递归构建）===
MyModel (root)
├── embedding: Embedding
├── block1: SubBlock
│   ├── linear: Linear
│   ├── bn: BatchNorm1d
│   └── relu: ReLU
├── block2: SubBlock
│   ├── linear: Linear
│   ├── bn: BatchNorm1d
│   └── relu: ReLU
├── dropout: Dropout
└── head: Linear


In [17]:
# 验证 _named_members 的 recurse 参数如何依赖 named_modules
# recurse=True → 调用 named_modules() → 获取所有后代模块的参数
# recurse=False → 仅获取当前模块的 _parameters 里的直接参数

print('=== recurse=True: 递归获取所有参数 (named_parameters 默认) ===')
params_recursive = list(model.named_parameters(recurse=True))
for name, param in params_recursive:
    print(f'  {name:40s} shape={list(param.shape)}')

print(f'\n=== recurse=False: 仅获取当前模块的直接参数 ===')
params_direct = list(model.named_parameters(recurse=False))
for name, param in params_direct:
    print(f'  {name:40s} shape={list(param.shape)}')

print(f'\n递归参数数: {len(params_recursive)}')
print(f'直接参数数: {len(params_direct)}')
print(f'MyModel 自身没有直接 Parameter（参数都在子模块里），所以 recurse=False 返回 0')
print(f'model._parameters 内容: {dict(model._parameters)}')

# 对有直接参数的子模块测试 recurse=False
print(f'\n--- 对 model.block1.linear 测试 recurse=False ---')
linear = model.block1.linear
print(f'block1.linear._parameters keys: {list(linear._parameters.keys())}')
for name, param in linear.named_parameters(recurse=False):
    print(f'  {name}: shape={list(param.shape)}')

=== recurse=True: 递归获取所有参数 (named_parameters 默认) ===
  embedding.weight                         shape=[1000, 128]
  block1.linear.weight                     shape=[256, 128]
  block1.linear.bias                       shape=[256]
  block1.bn.weight                         shape=[256]
  block1.bn.bias                           shape=[256]
  block2.linear.weight                     shape=[512, 256]
  block2.linear.bias                       shape=[512]
  block2.bn.weight                         shape=[512]
  block2.bn.bias                           shape=[512]
  head.weight                              shape=[10, 512]
  head.bias                                shape=[10]

=== recurse=False: 仅获取当前模块的直接参数 ===

递归参数数: 11
直接参数数: 0
MyModel 自身没有直接 Parameter（参数都在子模块里），所以 recurse=False 返回 0
model._parameters 内容: {}

--- 对 model.block1.linear 测试 recurse=False ---
block1.linear._parameters keys: ['weight', 'bias']
  weight: shape=[256, 128]
  bias: shape=[256]


### 实验 6：对比 `named_modules()` 和 `named_parameters()` 的前缀命名规则

In [18]:
# named_parameters 的参数名 = 模块前缀 + "." + 参数本地名
# 例如 "block1.linear.weight" → 模块前缀 = "block1.linear", 参数本地名 = "weight"

print('=== 模块名 vs 参数名前缀对应关系 ===')
params_by_module = {}
for name, param in model.named_parameters():
    parts = name.rsplit('.', 1)
    if len(parts) == 2:
        module_prefix, param_local = parts
    else:
        module_prefix, param_local = '', parts[0]
    
    if module_prefix not in params_by_module:
        params_by_module[module_prefix] = []
    params_by_module[module_prefix].append(param_local)

for mod_name, param_names in sorted(params_by_module.items()):
    display_name = f'"{mod_name}"' if mod_name else '"(root)"'
    print(f'  模块 {display_name:30s} 的参数: {param_names}')

# 验证：每个 named_modules 的模块的 _parameters 就是对应的参数
print(f'\n=== 交叉验证 ===')
for mod_name, mod in model.named_modules():
    direct_params = list(mod._parameters.keys())
    if direct_params:
        print(f'  "{mod_name or "(root)"}"._parameters = {direct_params}')

=== 模块名 vs 参数名前缀对应关系 ===
  模块 "block1.bn"                    的参数: ['weight', 'bias']
  模块 "block1.linear"                的参数: ['weight', 'bias']
  模块 "block2.bn"                    的参数: ['weight', 'bias']
  模块 "block2.linear"                的参数: ['weight', 'bias']
  模块 "embedding"                    的参数: ['weight']
  模块 "head"                         的参数: ['weight', 'bias']

=== 交叉验证 ===
  "embedding"._parameters = ['weight']
  "block1.linear"._parameters = ['weight', 'bias']
  "block1.bn"._parameters = ['weight', 'bias']
  "block2.linear"._parameters = ['weight', 'bias']
  "block2.bn"._parameters = ['weight', 'bias']
  "head"._parameters = ['weight', 'bias']


---

## Part D: 小结

### 要点清单

- [x] `named_modules()` 是层级遍历的核心引擎，使用 **DFS pre-order** 递归遍历
- [x] **pre-order** 含义：先产出当前节点，再递归产出子节点
- [x] `modules()` = `named_modules()` 去掉名字的薄包装
- [x] `children()` / `named_children()` 只遍历 `self._modules.items()`，**不递归**
- [x] **关键区别**：`modules()` 包含根模块自身；`children()` 不包含自身
- [x] **去重机制**：`memo` set 记录已产出的模块 `id()`，共享子模块只返回一次
- [x] `get_submodule(target)` 按 `.` 分割路径，逐层 `getattr` 查找，O(depth) 时间复杂度
- [x] `_named_members` 的 `recurse` 参数控制是否调用 `named_modules()` 递归获取所有后代的参数

### 方法速查表

| 方法 | 递归 | 包含自身 | 带名字 | 典型用途 |
|------|------|---------|--------|----------|
| `children()` | ❌ | ❌ | ❌ | 遍历顶层子模块 |
| `named_children()` | ❌ | ❌ | ✅ | 按名访问顶层子模块 |
| `modules()` | ✅ | ✅ | ❌ | 权重初始化、冻结 |
| `named_modules()` | ✅ | ✅ | ✅ | 按名操作子模块 |
| `get_submodule()` | — | ✅ (path="") | — | 按路径精确查找 |

### 与后续 Notebook 的关联

- **Notebook 02** 中 `_named_members` 的 `recurse` 参数直接依赖本 Notebook 的 `named_modules()`
- **Notebook 04** 中的 `_apply` 引擎使用 `self.modules()` 来递归对子模块应用转换函数
- **Notebook 06** 的 hook 系统执行顺序也遵循 DFS pre-order 遍历
- **Notebook 09** 的容器类（Sequential、ModuleList）有自定义的 `__getitem__` 索引访问

### 延伸阅读

- [PyTorch 源码 named_modules](https://github.com/pytorch/pytorch/blob/main/torch/nn/modules/module.py#L2300)
- [PyTorch 源码 get_submodule](https://github.com/pytorch/pytorch/blob/main/torch/nn/modules/module.py#L2400)
- [Composite 设计模式 (Wikipedia)](https://en.wikipedia.org/wiki/Composite_pattern)